In [1]:
# compute character metrics (per-character profiles)

# for each character in each doc: tanh(CharTr / (CharRe + ε))
# then aggregate across characters (and optionally across docs)

In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [3]:
MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']
PROMPTS = ['original', 'large']
COREF_DIR = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/models/linkappend/data-out/conll_to_json')
CHARACTERS_FILE = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/data/sampled_60/sampled_60_stories.json')

In [4]:
def load_jsonlines(filepath):
    documents = []
    with open(filepath, 'r') as f:
        for line in f:
            documents.append(json.loads(line))
    return documents

def clean_human_original_data(doc):
    # human original data has [SENT] in the end, need to remove it, an artefact from some processing
    if len(doc['sentences']) > 0:
        last_sent = doc['sentences'][-1]
        if len(last_sent) >= 3 and last_sent[-3:] == ['[', 'SENT', ']']:
            doc['sentences'][-1] = last_sent[:-3]
            num_tokens_before_last = sum(len(sent) for sent in doc['sentences'][:-1])
            removed_token_start = num_tokens_before_last + len(last_sent) - 3
            new_clusters = []
            for cluster in doc['clusters']:
                new_mentions = [m for m in cluster if m[0] < removed_token_start]
                if new_mentions:
                    new_clusters.append(new_mentions)
            doc['clusters'] = new_clusters
    return doc

def extract_story_id(doc_key):
    # formatting of story id
    if isinstance(doc_key, int):
        return doc_key
    s = str(doc_key).replace('story_', '')
    parts = s.split('_')
    if len(parts) >= 2 and parts[0] == 'doc':
        s = parts[1]
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None

In [5]:
def load_character_stories(filepath):
    # load character annotations for each story
    data = []
    valid_story_ids = set()
    with open(filepath, 'r') as f:
        for line in f:
            entry = json.loads(line)
            story_id = entry['story_id']
            characters = entry.get('characters', [])
            data.append({'story_id': story_id, 'characters': characters})
            has_chars = False
            for char in characters:
                if isinstance(char, dict) and 'name' in char:
                    name = char['name']
                    if name and name != '{}':
                        has_chars = True
                        break
                elif isinstance(char, str) and char and char != '{}':
                    has_chars = True
                    break
            if has_chars:
                valid_story_ids.add(story_id)
    return pd.DataFrame(data), valid_story_ids

def extract_flattened_tokens_and_boundaries(sentences):
    flat_tokens = []
    boundaries = []
    index = 0
    for sent in sentences:
        flat_tokens.extend(sent)
        end = index + len(sent) - 1
        boundaries.append((index, end))
        index += len(sent)
    return flat_tokens, boundaries

def get_character_sentence_presence(doc, character_stories_df):
    sentences = doc.get('sentences', [])
    N = len(sentences)
    flat_tokens, sentence_boundaries = extract_flattened_tokens_and_boundaries(sentences)
    story_id = extract_story_id(doc['doc_key'])

    match = character_stories_df[character_stories_df['story_id'] == story_id]
    if len(match) == 0:
        return {}, N

    characters = match.iloc[0]['characters']
    char_names = []
    for char in characters:
        if isinstance(char, dict) and 'name' in char:
            name = char['name']
            if name and name != '{}':
                char_names.append(name.lower())
        elif isinstance(char, str) and char and char != '{}':
            char_names.append(char.lower())
    char_names = sorted(list(set(char_names)))
    if not char_names:
        return {}, N

    char_sentences = {name: set() for name in char_names}
    clusters = doc.get('clusters', [])

    for cluster in clusters:
        cluster_sentences = set()
        mention_texts = []

        for mention in cluster:
            start_token_idx, end_token_idx = mention
            if end_token_idx < len(flat_tokens):
                mention_texts.append(' '.join(flat_tokens[start_token_idx:end_token_idx + 1]).lower())
                for sent_idx, (sent_start, sent_end) in enumerate(sentence_boundaries):
                    if sent_start <= start_token_idx <= sent_end:
                        cluster_sentences.add(sent_idx)
                        break

        if not cluster_sentences or not mention_texts:
            continue

        # if a character name appears in any mention text, attribute this cluster's sentence span to that character
        for char_name in char_names:
            if any(char_name in mtxt for mtxt in mention_texts):
                char_sentences[char_name].update(cluster_sentences)

    # keep only characters that were actually linked in coref mentions
    char_sentences = {k: v for k, v in char_sentences.items() if len(v) > 0}
    return char_sentences, N

EPSILON = 1e-8

def compute_character_metrics_per_character(doc, character_stories_df):
    char_sentences, N = get_character_sentence_presence(doc, character_stories_df)
    if N < 2 or len(char_sentences) == 0:
        return []

    rows = []
    for char_name, sent_set in char_sentences.items():
        presence = [1 if s in sent_set else 0 for s in range(N)]
        transfer = []
        for s in range(N - 1):
            transfer.append(1 if (presence[s] == 1 and presence[s + 1] == 1) else 0)
        CharTr_char = np.mean(transfer) if transfer else 0.0

        s_min = min(sent_set)
        s_max = max(sent_set)
        CharRe_char = (s_max - s_min) / (N - 1) if N > 1 else 0.0

        # char_coherence = tanh(CharTr / (CharRe + ε))
        # higher values indicate stronger local continuity relative to global spread
        char_coherence_char = np.tanh(CharTr_char / (CharRe_char + EPSILON))
        
        rows.append({
            'character_name': char_name,
            'CharTr': CharTr_char,
            'CharRe': CharRe_char,
            'char_coherence': char_coherence_char
        })
    return rows

In [6]:
def load_character_data(missing_policy='nan'):
    # missing_policy='zero' -> stories without valid chars are represented by one pseudo-character with 0
    # missing_policy='nan'  -> stories without valid chars are represented by one pseudo-character with NaN
    if missing_policy not in {'zero', 'nan'}:
        raise ValueError("missing_policy must be 'zero' or 'nan'")

    character_stories_df, valid_story_ids = load_character_stories(CHARACTERS_FILE)
    all_data = []

    for model in MODELS:
        for prompt in PROMPTS:
            if model == 'human' and prompt == 'large':
                seeds_to_load = ['seed42', 'seed43', 'seed44']
            else:
                seeds_to_load = ['seed42']

            for seed in seeds_to_load:
                filepath = COREF_DIR / f"{model}_{prompt}_{seed}_test_snapshots__local_json-nopound_pred.jsonlines"
                if not filepath.exists():
                    print(f"  Warning: {filepath.name} not found")
                    continue

                documents = load_jsonlines(filepath)
                for doc in documents:
                    if model == 'human' and prompt == 'original':
                        doc = clean_human_original_data(doc)

                    story_id = extract_story_id(doc['doc_key'])

                    # no valid annotation at story level
                    if story_id not in valid_story_ids:
                        fill_val = 0.0 if missing_policy == 'zero' else np.nan
                        all_data.append({
                            'model': model, 'prompt': prompt, 'seed': seed,
                            'story_id': story_id, 'character_name': '__NO_ANNOTATION__',
                            'CharTr': fill_val, 'CharRe': fill_val, 'char_coherence': fill_val
                        })
                        continue

                    char_rows = compute_character_metrics_per_character(doc, character_stories_df)
                    if len(char_rows) == 0:
                        # valid story id but no mapped character-coref match in this document
                        fill_val = 0.0 if missing_policy == 'zero' else np.nan
                        all_data.append({
                            'model': model, 'prompt': prompt, 'seed': seed,
                            'story_id': story_id, 'character_name': '__NO_CHARACTER_MATCH__',
                            'CharTr': fill_val, 'CharRe': fill_val, 'char_coherence': fill_val
                        })
                        continue

                    for row in char_rows:
                        all_data.append({
                            'model': model, 'prompt': prompt, 'seed': seed,
                            'story_id': story_id, **row
                        })

    return pd.DataFrame(all_data)

def prepare_char_data(df):
    # per-character output (human large averaged across seeds by story_id + character_name)
    results = []
    for model in MODELS:
        for prompt in PROMPTS:
            df_mp = df[(df['model'] == model) & (df['prompt'] == prompt)]
            if len(df_mp) == 0:
                continue

            if model == 'human' and prompt == 'large':
                df_avg = (
                    df_mp.groupby(['story_id', 'character_name'])[['CharTr', 'CharRe', 'char_coherence']]
                    .mean()
                    .reset_index()
                )
                for _, row in df_avg.iterrows():
                    results.append({
                        'model': model, 'prompt': prompt,
                        'story_id': row['story_id'], 'character_name': row['character_name'],
                        'CharTr': row['CharTr'], 'CharRe': row['CharRe'],
                        'char_coherence': row['char_coherence']
                    })
            else:
                df_seed42 = df_mp[df_mp['seed'] == 'seed42']
                for _, row in df_seed42.iterrows():
                    results.append({
                        'model': model, 'prompt': prompt,
                        'story_id': row['story_id'], 'character_name': row['character_name'],
                        'CharTr': row['CharTr'], 'CharRe': row['CharRe'],
                        'char_coherence': row['char_coherence']
                    })

    result = pd.DataFrame(results)
    result['story_id'] = result['story_id'].astype(int)
    return result

def character_to_story_view(df_characters):
    # derived story-level view: average over characters within each story
    story_df = (
        df_characters.groupby(['model', 'prompt', 'story_id'])[['CharTr', 'CharRe', 'char_coherence']]
        .mean()
        .reset_index()
    )
    return story_df

In [7]:
df_char_raw = load_character_data(missing_policy='nan')
print(df_char_raw.head())

   model    prompt    seed  story_id          character_name  CharTr  CharRe  \
0  human  original  seed42     13683                    bill     1.0     1.0   
1  human  original  seed42     13683                     tom     1.0     1.0   
2  human  original  seed42     13596  __NO_CHARACTER_MATCH__     NaN     NaN   
3  human  original  seed42     12423                    jeff     0.4     0.8   
4  human  original  seed42     12423                    matt     0.4     0.4   

   char_coherence  
0        0.761594  
1        0.761594  
2             NaN  
3        0.462117  
4        0.761594  


In [8]:
df_char = prepare_char_data(df_char_raw)

In [9]:
df_char.head()

,model,prompt,story_id,character_name,CharTr,CharRe,char_coherence
0,human,original,13683,bill,1.0,1.0,0.761594
1,human,original,13683,tom,1.0,1.0,0.761594
2,human,original,13596,__NO_CHARACTER_MATCH__,NaN,NaN,NaN
3,human,original,12423,jeff,0.4,0.8,0.462117
4,human,original,12423,matt,0.4,0.4,0.761594


In [10]:
# CharTr (character transfer): for each character, fraction of adjacent sentence pairs where character appears in both; higher means stronger local continuity

# CharRe (character spread): for each character, span of appearances across the story normalized by (N-1) sentences; higher means broader global spread

# char_coherence (per character) = tanh(CharTr / (CharRe + ε))
# high: dense local continuity relative to span
# low: weaker continuity relative to spread
# ε = 1e-8 to avoid division by zero when CharRe = 0

In [11]:
char_agg = (
    df_char.groupby(['model', 'prompt'])
    .agg(
        CharTr=('CharTr', 'mean'),
        CharRe=('CharRe', 'mean'),
        char_coherence_mean=('char_coherence', 'mean'),
        char_coherence_std=('char_coherence', 'std'),
        observed_rows=('char_coherence', 'count'),  # non-NaN rows used in mean
        total_rows=('char_coherence', 'size')       # all rows in group
    )
    .reset_index()
    .assign(missing_policy='zero', unit='character')
)
char_agg

,model,prompt,CharTr,CharRe,char_coherence_mean,char_coherence_std,observed_rows,total_rows,missing_policy,unit
0,claude45,large,0.260770,0.519048,0.346036,0.323994,106,119,zero,character
1,claude45,original,0.304592,0.560329,0.384856,0.326027,105,118,zero,character
2,gpt4o,large,0.345738,0.585374,0.423709,0.320619,119,132,zero,character
3,gpt4o,original,0.397135,0.608142,0.432632,0.335082,118,131,zero,character
4,human,large,0.467764,0.639034,0.488189,0.261702,124,146,zero,character
5,human,original,0.458802,0.659909,0.532168,0.285911,99,114,zero,character
6,internvl3,large,0.314148,0.621237,0.359780,0.308091,112,127,zero,character
7,internvl3,original,0.495742,0.757475,0.453553,0.323273,111,126,zero,character
8,llama4scout,large,0.335192,0.549029,0.393881,0.329199,76,105,zero,character
9,llama4scout,original,0.324363,0.419000,0.396729,0.362570,33,81,zero,character


In [12]:
# so now means are comparable, because for every model/prompt we take into account all mentioned characters
# humans remain strongest on char coherence score, they have the highest continuity-relative-to-spread behaviour
# gpt4o is close to humans
# llama4scout is the weakest across all three

# coherence is computed per character, then averaged across characters
# that's why a model can have high individual metric, but not highest coherence mean

In [13]:
char_agg = (
    df_char.groupby(['model', 'prompt'])
    .agg(
        CharTr=('CharTr', 'mean'),
        CharRe=('CharRe', 'mean'),
        char_coherence_mean=('char_coherence', 'mean'),
        char_coherence_std=('char_coherence', 'std'),
        observed_rows=('char_coherence', 'count'),  # non-NaN rows used in mean
        total_rows=('char_coherence', 'size')       # all rows in group
    )
    .reset_index()
    .assign(missing_policy='nan', unit='character')
)
char_agg

,model,prompt,CharTr,CharRe,char_coherence_mean,char_coherence_std,observed_rows,total_rows,missing_policy,unit
0,claude45,large,0.260770,0.519048,0.346036,0.323994,106,119,nan,character
1,claude45,original,0.304592,0.560329,0.384856,0.326027,105,118,nan,character
2,gpt4o,large,0.345738,0.585374,0.423709,0.320619,119,132,nan,character
3,gpt4o,original,0.397135,0.608142,0.432632,0.335082,118,131,nan,character
4,human,large,0.467764,0.639034,0.488189,0.261702,124,146,nan,character
5,human,original,0.458802,0.659909,0.532168,0.285911,99,114,nan,character
6,internvl3,large,0.314148,0.621237,0.359780,0.308091,112,127,nan,character
7,internvl3,original,0.495742,0.757475,0.453553,0.323273,111,126,nan,character
8,llama4scout,large,0.335192,0.549029,0.393881,0.329199,76,105,nan,character
9,llama4scout,original,0.324363,0.419000,0.396729,0.362570,33,81,nan,character


In [14]:
# under nan policy, meaning that means are computed only on stories where characters are present, we get the following
# humans are strongest overall on the char coherence in both large and original
# intern vl original has the highest CharTr and CharRe, but not the highest coherence mean
# gpt4o is consistently close to humans

# the computation here is made only when there is a character, so
# this is why means are computed on different effective samples, the reporting above looks at zero policy reporting (assign 0 when character is not present), making sure that numbers are computed on the same number of rows

In [17]:
chardata_path = Path('./analysis_data/character/')
chardata_path.mkdir(parents=True, exist_ok=True)

for policy in ['zero', 'nan']:
    df_raw_policy = load_character_data(missing_policy=policy)
    df_char_policy = prepare_char_data(df_raw_policy)

    char_agg_policy = (
        df_char_policy.groupby(['model', 'prompt'])
        .agg(
            CharTr=('CharTr', 'mean'),
            CharRe=('CharRe', 'mean'),
            char_coherence_mean=('char_coherence', 'mean'),
            char_coherence_std=('char_coherence', 'std'),
            observed_rows=('char_coherence', 'count'),
            total_rows=('char_coherence', 'size')
        )
        .reset_index()
        .assign(missing_policy=policy, unit='character')
    )

    df_raw_policy.to_csv(chardata_path / f'character_metrics_raw_{policy}_policy.csv', index=False)
    df_char_policy.to_csv(chardata_path / f'character_metrics_per_character_{policy}_policy.csv', index=False)
    char_agg_policy.to_csv(chardata_path / f'character_metrics_agg_per_character_{policy}_policy.csv', index=False)

print('Saved character metric files for policies: zero, nan')

Saved character metric files for policies: zero, nan
